# 15 — Imbalance-XGBoost

Import trực tiếp gradient/Hessian weighted và focal từ commit tác giả, chạy qua API xgb.train hiện tại. So với XGB không weight cùng preprocessing/budget. Early stopping theo unweighted validation AP.

Không sửa cấu hình ở notebook này. Muốn đổi cấu hình hãy tạo campaign mới ở11.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "baf.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BAF_project hoặc BAF_project/notebooks.")

CACHE_ROOT = Path(
    os.environ.get("BAF_CACHE_ROOT") or ROOT / ".cache" / "baf-research"
).expanduser().resolve()
for name, subdirectory in {
    "PIP_CACHE_DIR": "pip",
    "TORCH_HOME": "torch",
    "XDG_CACHE_HOME": "misc",
    "TEMP": "temp",
    "TMP": "temp",
}.items():
    folder = CACHE_ROOT / subdirectory
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(folder)

sys.path.insert(0, str(ROOT / "src"))
import gru

print("Python:", sys.executable)
print("Project:", ROOT)
print("Module:", gru.RESEARCH_VERSION)


Python: d:\UIT\.venvs\ml-gpu\Scripts\python.exe
Project: D:\UIT\DS317\BAF_project
Module: baf-research-20261028-v1


In [2]:
# Điền đường dẫn in ở notebook 11 nếu đã có nhiều campaign.
# Ví dụ: Path(r"D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_...")
CAMPAIGN_DIR = None
CAMPAIGN = gru.research_resolve_campaign(ROOT, CAMPAIGN_DIR)
PLAN = gru.research_read(CAMPAIGN / "campaign.json")

print("Campaign:", CAMPAIGN)
print("Mode:", PLAN["mode"])
print("Suites:", PLAN["suites"])
print("Seeds:", PLAN["config"]["seeds"])


Campaign: D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_20261028_214637_b9a2294e
Mode: full
Suites: ['losses', 'meta', 'lsr', 'xgboost']
Seeds: [42, 317, 2026]


## 1. Chạy training có notebook output thật

        Cell khởi chạy một Python process mới bằng đúng interpreter của kernel. Process thực thi từng cell trong notebook con, lưu stdout/display/error sau từng cell và in log trực tiếp bên dưới. Fit hoàn tất được kiểm hash rồi reuse khi resume. Nếu bị ngắt, không xóa campaign. Cell này có thể chạy lâu.

In [3]:
SUITE = 'xgboost'
RUN_DIR = None
if SUITE in PLAN["suites"]:
    RUN_DIR = gru.research_launch_suite(CAMPAIGN, SUITE)
else:
    print("SKIPPED: suite không thuộc phạm vi đã chọn ở11:", SUITE)


BEGIN_STAGE 1
END_STAGE 1
BEGIN_STAGE 2
RESEARCH_STUDY xgboost mode= full device= cuda
END_STAGE 2
BEGIN_STAGE 3
FIT_BEGIN xgb_none_c00_s42 {'method': 'xgb_none', 'family': 'xgboost', 'candidate': 0, 'depth': 3, 'gamma': 2.0, 'alpha': 3.1468023773429072, 'feature': 'dense'}
[0]	validation-unweighted_AP:0.06264
[25]	validation-unweighted_AP:0.09501
[50]	validation-unweighted_AP:0.10371
[75]	validation-unweighted_AP:0.12247
[100]	validation-unweighted_AP:0.13742
[125]	validation-unweighted_AP:0.14934
[150]	validation-unweighted_AP:0.15702
[175]	validation-unweighted_AP:0.16246
[200]	validation-unweighted_AP:0.16768
[225]	validation-unweighted_AP:0.17136
[250]	validation-unweighted_AP:0.17399
[275]	validation-unweighted_AP:0.17647
[299]	validation-unweighted_AP:0.17774
FIT_COMPLETE xgb_none_c00_s42 validation_AP= 0.17773585119202398
FIT_BEGIN xgb_none_c01_s42 {'method': 'xgb_none', 'family': 'xgboost', 'candidate': 1, 'depth': 5, 'gamma': 2.0, 'alpha': 9.902365202050971, 'feature': 'dense

## 2. Xem kết quả validation đã khóa

        Chưa có test metric ở notebook này. Model, configuration và threshold được lưu trước khi mở test ở16.

In [4]:
if RUN_DIR is not None:
    display(gru.research_suite_overview(RUN_DIR))
    print("Notebook training thật:")
    for executed in sorted((RUN_DIR / "attempts").glob("*/training_executed.ipynb")):
        print(executed)


,method,seed,validation_ap,threshold,fit_seconds,family,feature
0,xgb_focal,42,0.180428,0.235303,57.431179,xgboost,dense
1,xgb_focal,317,0.179369,0.235457,57.135962,xgboost,dense
2,xgb_focal,2026,0.178617,0.236091,56.638009,xgboost,dense
3,xgb_none,42,0.181503,0.042400,26.000013,xgboost,dense
4,xgb_none,317,0.179706,0.041689,31.083082,xgboost,dense
5,xgb_none,2026,0.181536,0.041641,24.477419,xgboost,dense
6,xgb_weighted,42,0.179823,0.295406,32.237048,xgboost,dense
7,xgb_weighted,317,0.181974,0.294826,32.176602,xgboost,dense
8,xgb_weighted,2026,0.182595,0.296123,30.739584,xgboost,dense


Notebook training thật:
D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_20261028_214637_b9a2294e\suites\xgboost\attempts\20261029_084231_4aa6084a\training_executed.ipynb
D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_20261028_214637_b9a2294e\suites\xgboost\attempts\20261029_085039_e1592eda\training_executed.ipynb


## 3. Xuất proof trong một lượt

        Không cần Ctrl+S hoặc chạy cell cuối hai lần. Full proof giữ trên máy; review nhỏ hơn để gửi, ghi rõ phần đã bỏ. Notebook `verify.ipynb` được execute thật, kiểm hash/score và không fit lại.

In [5]:
if RUN_DIR is not None:
    PROOF = gru.research_export_proof(RUN_DIR, ROOT)
    for kind, path in PROOF.items():
        print(kind, "→", path)


RESEARCH_FROZEN_SUITE_VERIFIED 9
FULL_PROOF D:\UIT\DS317\BAF_project\results\proofs\xgboost_b45eaaa3_full_proof.zip
PARTIAL_REVIEW D:\UIT\DS317\BAF_project\results\proofs\xgboost_b45eaaa3_review.zip
full_proof → D:\UIT\DS317\BAF_project\results\proofs\xgboost_b45eaaa3_full_proof.zip
review → D:\UIT\DS317\BAF_project\results\proofs\xgboost_b45eaaa3_review.zip
